# <font color="purple">**Greek ERGANI XML Data Validation Tool**</font>
This notebook breaks down the pre-submission auditing tool used to verify employee daily work schedule XML against ERGANI compliance standards. It is designed to log every single missing data element alongside its exact physical line number in the XML file.

### <font color="blue">**Step 1: Load Dependencies**</font>
The first step is to import the necessary standard modules for handling web requests, parsing XML structures and using regular expressions (`urllib`, `ElementTree`, `re`).

In [41]:
import urllib.request, urllib.parse, urllib.error
import xml.etree.ElementTree as ET
import re

### <font color="blue"> **Step 2: Fetch & Decode Raw Data**</font>
After requesting the user to provide the URL of the raw XML file, urllib is used to open, read and decode the file, and then, using `.splitlines()`, the file is converted to a list of elements for future processing. Each element of the list is a line of the XML file.

<font color="red">**NOTE:**</font> If the user does not provide the Python tool with a URL, the file 'Work Schedule 22-23.05.2026.xml' will be used by default.

In [42]:
# Request the  raw XML file.
url=input('Enter the URL of your raw XML file (or press ENTER for default): ')
if len(url)<1:
   url='https://raw.githubusercontent.com/kyreugenia/Data_Analysis_Portfolio/refs/heads/main/ERGANI_XML_Data%20Validator/Work%20Schedule%2022-23.05.2026.xml'

# Open and read the XML file as a list of plain lines.
link=urllib.request.urlopen(url)
line_list = link.read().decode().splitlines()
print (f'\033[1;95mFor demonstration purposes, the first three elements of the produced list will look like this:\033[0m\n{line_list[:3]}')

For demonstration purposes, the first three elements of the produced list will look like this:
['<?xml version="1.0" encoding="utf-8"?>', '<WTOS xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.yeka.gr/WTO">', '  <WTO xmlns="">']


### <font color="blue"> **Step 3: Inject Line Numbers**</font>
This step focuses on determining the line of every key tag in the XML file. 

Noticing that all the key tags have a predictable pattern (starting  with  `f_` and closing with `>`), it makes it easy to locate them using `regular expressions`.

Then, leveraging the `re.sub()` function, he script automatically replaces the standard opening tag with an updated version that embeds a custom `line_num` attribute.  

Once the process in complete, the updated `line_list` is concatenated into a valid XML string format (by joining the lines sequentially using `\n`) and then parsed into the XML tree using the `fromsting()` function of **ElementTree**.

In [43]:
# Add line number attribute.
for line in range(len(line_list)):
   if '<f_' in line_list[line]:
      line_list[line]=re.sub(r'<f_(\w+)>', r'<f_\1 line_num="' + str(line+1) + '">', line_list[line])

# Parse the modified text into the XML tree.
tree = ET.fromstring("\n".join(line_list))

In [44]:
# Find all potential tags.
element_list=tree.findall('.//ErgazomenoiWTO/')+tree.findall('.//ErgazomenosWTOAnalytics/')
print ('\nStarting XML Validation Report...\n')

# Map tags to readable names for reporting.
tags= {'f_afm':'AFM','f_eponymo':'surname','f_onoma':'name','f_date':'date','f_type':'type','f_from': 'start hour','f_to':'end hour'}

# Check the AFMs, names, surnames, dates, type and time for missing elements.
index=None
for index,element in enumerate(element_list):
   # Skip the tags 'ErgazomenosAnalytics' and 'ErgazomenosWTOAnalytics' since they do not need checking.
   if element.tag=='ErgazomenosAnalytics'or element.tag=='ErgazomenosWTOAnalytics':
      continue

   # Skip time checks entirely if the shift type is "ΑΝ" (Rest/Absence).
   if element.tag=='f_from' and (element_list[index-1]).text=="ΑΝ":
      continue
   if element.tag=='f_to' and (element_list[index-2]).text=="ΑΝ":
      continue

   #Check for missing elements.
   if element.text is None or element.text.strip()=='':
      print ('Missing',tags[element.tag],'in line',element.get('line_num'))
      continue

   # Add an extra check for AFM to ensure it is 9 digit long.
   if element.tag=='f_afm': 
      if element.text is not None and len(element.text)!=9: 
         print('Wrong AFM in line',element.get('line_num'))
      





Starting XML Validation Report...

Wrong AFM in line 12
Wrong AFM in line 25
Missing name in line 95
Missing date in line 109
Missing AFM in line 120
Missing surname in line 161
Wrong AFM in line 471
Missing type in line 139
Missing start hour in line 140
Missing end hour in line 141
Missing type in line 1261
Missing start hour in line 1262
Missing end hour in line 1263
